# 02 — LLaMEA Synthesis

**f1–f24 · 10D · Explicit Baseline · heteroscedastic noise σ = 0.2**

Saved outputs may reflect earlier runs. Use separate study storage for an independent experiment.


In [1]:
import os
import sys
from pathlib import Path

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(root_dir)
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Keep module identities stable for multiprocessing/pickling.
# After updating project code or configuration, restart the kernel instead of evicting modules.

from evolution.application.model_selection import ModelSelection
from evolution.infra.di import EvolutionProvider
from evolution.application.campaign.audit import CampaignAudit
from evolution.application.campaign.service import SynthesisCampaign
from shared.infra.lifecycle import WorkflowScope

print("[OK] LLaMEA Evolution Pipeline initialized.")

# Select the provider and exact model for this campaign.
model = ModelSelection(provider="upb", model_id="gwdg.qwen3-coder-next")

# To use a local server, edit both fields, for example:
# model = ModelSelection(provider="local", model_id="qwen2.5-coder-32b-instruct-q4_k_m.gguf")


[OK] LLaMEA Evolution Pipeline initialized.


## 1. Connect the model


In [2]:
# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(EvolutionProvider(), context={}, request_context={})
campaign = services.get(SynthesisCampaign)
audit = services.get(CampaignAudit)
print("=" * 65)
print(f"Provider: {model.provider}")
print(f"Model ID: {model.model_id}")
print("=" * 65)

Diagnostic log: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/logs/evolution/2026-11-20_15-36-10_UTC-1157439b/diagnostics.log


Provider: upb
Model ID: gwdg.qwen3-coder-next


## 2. Audit synthesis coverage

Check the selected matrix against the current database before running.


In [3]:
matrix_df_result = audit.inspect(
    model=model,
    problem_targets=[{"id": problem, "dimensions": [10]} for problem in range(1, 25)],
    noise_conditions=[{"std": 0.2, "noise_model": "heteroscedastic"}],
    synthesis_modes=[{"mode": "explicit", "strategies": ["baseline"]}],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
)
if matrix_df_result.is_success:
    matrix_df, summary = matrix_df_result.value
    display(matrix_df) if "display" in globals() else print(matrix_df.to_string())
else:
    matrix_df = None
    summary = None
    print(f"Error: {matrix_df_result.error.message}")



🎯 Synthesis Matrix Audit for 'gwdg.qwen3-coder-next':
   • Configured Matrix Size:  24 conditions
   • Completed (Champions):   23/24 (95.8%)
                                                             Target Runs  Completed  Running Records  Failed Records               Status
Problem                   Dimension Environment    Strategy                                                                              
f1 (Sphere)               10D       Explicit (0.2) Baseline            1          2                0               0           ✅ Complete
f2 (Ellipsoidal)          10D       Explicit (0.2) Baseline            1          1                0               0           ✅ Complete
f3 (Rastrigin)            10D       Explicit (0.2) Baseline            1          1                0               0           ✅ Complete
f4 (Büche-Rastrigin)      10D       Explicit (0.2) Baseline            1          1                0               0           ✅ Complete
f5 (Linear)               10D

## 3. Run synthesis

Completed conditions are skipped; interrupted runs require their checkpoints to resume.


In [ ]:
# Run the selected f1–f24 matrix; completed conditions follow the existing resume/skip rules.
results_result = campaign.run(
    model=model,
    problem_targets=[{"id": problem, "dimensions": [10]} for problem in range(1, 25)],
    noise_conditions=[{"std": 0.2, "noise_model": "heteroscedastic"}],
    synthesis_modes=[{"mode": "explicit", "strategies": ["baseline"]}],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
)
results = results_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)



🧬 LLAMEA EVOLUTIONARY ALGORITHM SYNTHESIS
   Model: gwdg.qwen3-coder-next | Pending Tasks: 1 | Concurrency: 8 workers
⚠️  Replacing Exp ID: #7268: old iterations, champion, code and checkpoint will be removed.

[1/1] 🧬 Task: gwdg.qwen3-coder-next | Dim: 10D | Noise: noisy (σ=0.2) | Problem: f10 (EllipsoidRotated) | Strategy: Baseline | Exp ID: #7268
  • ⚡ Gen  1/10 | Algo: NaCMAES            | Error:   1.0419e+04 | Fitness:  -1.0419e+04 | Evals: 100,005 | Time:  0.41s
  • ⚡ Gen  2/10 | Algo: RobustCMAES        | Error:   2.6577e+03 | Fitness:  -2.6577e+03 | Evals: 100,002 | Time:  0.34s
  • ⚡ Gen  3/10 | Algo: StableNoisyCMA     | Error: FAILED (ValueError) | Time:  0.33s
  • ⚡ Gen  4/10 | Algo: StableNoisyCMA     | Error: FAILED (ValueError) | Time:  0.32s
  • ⚡ Gen  5/10 | Algo: StableNoisyCMA     | Error:   2.4734e+03 | Fitness:  -2.4734e+03 | Evals:  99,992 | Time:  0.46s


## Optional: one experiment in memory

Uncomment the example and run only the import cell and this example. Study data is not saved and cannot be resumed; operational logs and throttle state may persist.


In [ ]:
# from evolution.application.synthesis.service import Synthesis
# from evolution.infra.di import TransientEvolutionProvider
#
# transient_services = WorkflowScope(
#     TransientEvolutionProvider(), context={}, request_context={}
# )
# try:
#     single_synthesis = transient_services.get(Synthesis)
#     single_result = single_synthesis.run(
#         model=model,
#         problem_id=1,
#         dim=10,
#         noise_std=0.2,
#         noise_model="heteroscedastic",
#         mode="explicit",
#         prompt_strategy="baseline",
#         budget=100_000,
#         iterations=10,
#         timeout_seconds=30.0,
#         synthesis_seed=43,
#         instance_id=1,
#     )
#     if single_result.is_success:
#         print(f"Transient synthesis: best error = {single_result.value.best_error}")
#     else:
#         print(single_result.error.message)
# finally:
#     transient_services.close()
